<a href="https://colab.research.google.com/github/mariaaapetrovskaya/Developing-a-Model-for-Automatic-Gesture-Recognition/blob/main/0_%D0%B3%D0%B8%D0%BF%D0%BE%D1%82%D0%B5%D0%B7%D0%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install av

from transformers import AutoModel, CLIPImageProcessor, CLIPTokenizer
from datasets import load_dataset
import torch
from PIL import Image
from tqdm import tqdm
from collections import Counter


dataset = load_dataset("mapetrovska/gesturedataset")
model_name = "BAAI/EVA-CLIP-8B"
model = AutoModel.from_pretrained(model_name, trust_remote_code=True).eval()
tokenizer = CLIPTokenizer.from_pretrained(model_name)


num_vision_pos = model.vision_model.embeddings.position_embedding.num_embeddings
model.vision_model.embeddings.position_ids = torch.arange(num_vision_pos).unsqueeze(0)

num_text_pos = model.text_model.embeddings.position_embedding.num_embeddings
model.text_model.embeddings.position_ids = torch.arange(num_text_pos).unsqueeze(0)

device = next(model.parameters()).device
model.vision_model.embeddings.position_ids = model.vision_model.embeddings.position_ids.to(device)
model.text_model.embeddings.position_ids = model.text_model.embeddings.position_ids.to(device)


processor = CLIPImageProcessor(
    size={"shortest_edge": 224},
    crop_size={"height": 224, "width": 224},
    do_center_crop=True,
    do_normalize=True,
    do_resize=True,
    image_mean=[0.48145466, 0.4578275, 0.40821073],
    image_std=[0.26862954, 0.26130258, 0.27577711],
    resample=Image.BICUBIC,
)


text_labels = [str(i) for i in range(8)]
input_ids = tokenizer(text_labels, return_tensors="pt", padding=True).input_ids.to(device)

with torch.no_grad():
    text_features = model.encode_text(input_ids)
    text_features /= text_features.norm(dim=-1, keepdim=True)


def extract_middle_frame(video_decoder):
    frames = []
    for frame in video_decoder:
        if isinstance(frame, torch.Tensor):
            if frame.shape[0] == 3:
                frame = frame.permute(1, 2, 0)
            frame = Image.fromarray(frame.numpy().astype('uint8'))
        frames.append(frame)
    if not frames:
        return None
    return frames[len(frames) // 2]

correct = 0
total = 0

for sample in tqdm(dataset['train']):
    middle_frame = extract_middle_frame(sample['video'])
    if middle_frame is None:
        continue

    true_label = sample['label']
    input_pixels = processor(images=middle_frame, return_tensors="pt").pixel_values.to(device)

    with torch.no_grad():
        image_features = model.encode_image(input_pixels)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        probs = (100.0 * image_features @ text_features.T).softmax(dim=-1)
        pred_label = torch.argmax(probs[0]).item()

    if pred_label == true_label:
        correct += 1
    total += 1

print(f"Всего: {total}")
print(f"Правильных: {correct}")
print(f"Accuracy: {correct/total*100:.2f}%")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 18.9 MB/s eta 0:00:00


Resolving data files:   0%|          | 0/101 [00:00<?, ?it/s]

жесты_внутреннего_состояния/12XSAqMtRnse(…):   0%|          | 0.00/1.07M [00:00<?, ?B/s]

жесты_внутреннего_состояния/14Lt-Y9fwTkA(…):   0%|          | 0.00/318k [00:00<?, ?B/s]

дейктические_жесты/1TXuBNc6sASqpu8pIrAXC(…):   0%|          | 0.00/299k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/472 [00:00<?, ?B/s]


 27%|██▋       | 27/101 [1:49:55<5:26:55, 265.08s/it]